In [106]:
from scipy.constants import electron_mass

mol_dir = "./mols"
basis_sets_dir = "./basis-sets"

particle_properties_file = "particle-properties.json"

In [107]:
e_basis_set = "def2-SVP"
n_basis_set = "DZSPN"

mol_name = "LiH"

In [149]:
import numpy as np
import json
import itertools

from scipy.sparse import coo_matrix, csr_matrix
from scipy.linalg import block_diag

from pyscf import gto, scf
from pyscf.lo import orth

from gbasis.wrappers import from_pyscf
from gbasis.parsers import parse_nwchem
from gbasis.parsers import make_contractions

from gbasis.integrals.overlap import overlap_integral
from gbasis.integrals.kinetic_energy import kinetic_energy_integral
from gbasis.integrals.electron_repulsion import electron_repulsion_integral

# Load properties of all possible particles (spin, fermion/boson, mass, charge, etc)
with open(particle_properties_file, "r") as file:
    particle_properties = json.load(file)

# Build molecule for PySCF
mol = gto.Mole()
mol.atom = mol_dir + '/' + mol_name + '.xyz'
mol.basis = e_basis_set
mol.build()

mol_zs = mol.atom_charges()
mol_symbs = [mol.atom_symbol(i) for i in range(mol.natm)] # Atomic symbols
mol_coords = mol.atom_coords()

# Run restricted Hartree Fock to get better orbitals (I might truncate the highest energy ones)
hf = scf.RHF(mol).run() # TODO: apparently there might be better choices of orbital to allow for truncations (FNO). look into?

# Load basis dictionary (atomic orbitals) for nuclear orbitals
n_basis_dict = parse_nwchem(basis_sets_dir + '/nuclear/' + n_basis_set + '.nw')

# Construct a dictionary of all the particle types that will be in our calculation, along with their orbitals and info like spin.
# Note: we will use the order of the dictionary. Python 3.7+ guarantees when we iterate, the dictionary will be ordered according to when the elements were added.
particles = {}

for i in range(mol.natm):
    symb = mol.atom_symbol(i)
    # If this is a particle type (nucleus) we haven't seen before
    if symb not in particles:
        # Add it to particle list
        particles[symb] = {}
        particles[symb]['coords'] = []
        particles[symb]['count'] = 0

    # Add its coordinates to the list
    particles[symb]['coords'].append(mol_coords[i])

    particles[symb]['count'] += 1

for symb in particles:
    # GBasis wants coords as numpy array
    particles[symb]['coords'] = np.array(particles[symb]['coords'])

    # Construct a basis w/ GBasis for each of the nuclear particles
    particles[symb]['basis'] = make_contractions(n_basis_dict, # Basis of (nuclear) AOs to use
                                                 [symb] * particles[symb]['count'], # Types of atoms (all the same)
                                                 particles[symb]['coords'], # Coordinates
                                                 coord_types='cartesian')

    # Transform to orthonormal orbitals
    overlap = overlap_integral(particles[symb]['basis'])
    particles[symb]['transform'] = orth.lowdin(overlap) # Symmetric orthonormalization of AOs

    # Number of spatial orbitals
    particles[symb]['no_spatial_orbitals'] = overlap.shape[0]

# Retrieve some important properties on each particle (spin, fermion/boson, mass, charge)
for symb in particles:
    particles[symb]['properties'] = particle_properties[symb]

# Add electrons to our particle list
particles['e'] = {}
particles['e']['basis'] = from_pyscf(mol) # Gbasis set of GTOs (gaussian type orbitals) for electronic particles
particles['e']['transform'] = hf.mo_coeff.T # transform to MOs that will be used for calculation
particles['e']['count'] = mol.nelectron  # Get number of electrons from PySCF, truncate off 10
particles['e']['no_spatial_orbitals'] = hf.mo_coeff.shape[0] - 7

# Retrieve some important properties on each particle (spin, fermion/boson, mass, charge)
for symb in particles:
    particles[symb]['properties'] = particle_properties[symb]

    # Number of spin orbitals, since we now have the particle's spin
    particles[symb]['no_spin_orbitals'] = particles[symb]['no_spatial_orbitals'] * particles[symb]['properties']['spin']

# Amount of particles we have
particle_types = len(particles)

# List of particle names for easy indexing
particle_names = [symb for symb in particles]

converged SCF energy = -7.9786624829859


In [144]:
# TODO: I should probably just do all the integrals at once by combining the bases of all particles.

# Our basis in particles[symb]['basis'] is a basis of spatial orbitals, so the integrals will be between spatial oritals.
# This class will take a matrix (or 4D array for 2-body interactions) of these spatial integrals and give us a matrix that indexes spin orbitals
# We use the convention defined below for indexing spin orbitals
#
# This class ASSUMES 2-body integrals are in CHEMIST'S NOTATION:
# data[i,j,k,l] = \int dr_1 dr_2 \chi_i^*(r_1) \chi_j(r_1) \hat{O}_2 \chi_k^*(r_2) \chi_l(r_2)
class IntegralSpinWrapper:
    # data: table of integrals (2D array for 1-body, 4D array for 2-body). Should be a numpy array, which it will be for integrals from GBasis
    # spin: number of spin states if 1-body, or 2-tuple of the number of spin states for each particle if 2-body
    def __init__(self, data, spin):
        self.data = data
        self.spin = spin

        self.is_two_body = len(data.shape) == 4

    def __getitem__(self, index):
        # If one-body interaction
        if not self.is_two_body:
            # Spin states must be equal.
            if index[0] % self.spin == index[1] % self.spin:
                return self.data[int(index[0] / self.spin), int(index[1] / self.spin)]

            # Otherwise by orthonormality the integral is 0
            else:
                return 0

        # If two-body interaction
        else:
            # Spin states must be equal for both coordinates:
            if (index[0] % self.spin[0] == index[1] % self.spin[0]) and (index[2] % self.spin[1] == index[3] % self.spin[1]):
                return self.data[int(index[0]/self.spin[0]), int(index[1]/self.spin[0]), int(index[2]/self.spin[1]), int(index[3]/self.spin[1])]

            # Otherwise by orthonormality the integral is 0
            else:
                return 0

# Construct the contribution to the FCI Hamiltonian from a one-body interaction (e.g. <e_1 | KE | e_3> where e_i are electronic basis states)
# We can have any state for the remaining particles so we will iterate over all possible determinants for every other particle

# particle_no: index in the dictionary of the particle (use particle_names for indexing)
# bra: index of the bra state in particle particle_no (this is a determinant/permanent, not a single particle state!)
# ket: inex of the ket state                          (this is a determinant/permanent, not a single particle state!)
# value: value of <bra | O_1 | ket>
def construct_1_particle_interaction(particle_no, bra, ket, value):
    # The idea here is that the states (in our huge big space) that involve this specific matrix element <bra | O_1 | ket> are those that look like
    # bra: |  ANY  |  ANY  |  ...  |  bra  |  ... |  ANY  |
    # ket: |  ANY^ |  ANY^ |  ...  |  ket  |  ... |  ANY  |
    #        ptcl 1  ptcl 2   ..   ptcl ptcl_no ..    ptcl particle_types
    # where the boxes are our choice of determinant/permanent for each particle type, and between the bra and the ket, have to be the same for all
    # particle types other than the one labeled by ptcl_no (the argument, particle_no).
    # So we will iterate over all possible choices of determinants for the other particles

    # Lists of the row and column indices and the values (will all be the same, value) in the Hamiltonian matrix.
    mtx_rows = []
    mtx_cols = []
    mtx_values = []

    # Iterate all over choices of determinant/permanent for the particles coming BEFORE ptcl_no
    for i in range(bases[particle_no]):
        # Iterate all over choices of determinant/permanent for the particles coming AFTER ptcl_no
        for j in range(int(total_states/bases[particle_no+1])):
            # Construct the indices in the Hamiltonian matrix (base convention) for these bra and ket states
            bra_idx = i + (bra * bases[particle_no]) + (j * bases[particle_no+1])
            ket_idx = i + (ket * bases[particle_no]) + (j * bases[particle_no+1])

            # Add to the list of elements
            mtx_rows.append(bra_idx)
            mtx_cols.append(ket_idx)
            mtx_values.append(value)

    return coo_matrix((mtx_values, (mtx_rows, mtx_cols)), shape=(total_states, total_states)).tocsr()

# Construct the contribution to the FCI Hamiltonian from a two-body interaction between two DISTINCT PARTICLES(e.g. <e_1n_2 | V | e_3e_4> where e_i are one-particle electronic basis states and n_i are one-particle basis states for some other particle)
# We can have any state for the remaining particles so we will iterate over all possible determinants for every other particle
# Note that two-body interactions between particles of the same type should be handled by construct_1_particle
# NOTE: particle1_no MUST be less than particle2_no for this to work.

# particle1_no: index in the dictionary for particle type 1 (use particle_names for indexing)
# particle2_no: index in the dictionary for particle type 2
# bra1: index of the bra state in particle particle1_no (this is a determinant/permanent, not a single particle state!)
# ket1: inex of the ket state                           (this is a determinant/permanent, not a single particle state!)
# etc...
# value: value of <bra | O_1 | ket>
def construct_2_particle_interaction(particle1_no, particle2_no, bra1, ket1, bra2, ket2, value):
    # The idea here is that the states (in our huge big space) that involve this specific matrix element <bra1 | O_1 | ket> are those that look like
    # bra: |  ANY  |  ANY  |  ...  |  bra1  |  ... |  bra2  |  ... |  ANY  |
    # ket: |  ANY^ |  ANY^ |  ...  |  ket1  |  ... |  bra2  |  ... |  ANY  |
    #        ptcl 1  ptcl 2   ..   ptcl ptc1l_no ..    ptcl2_no particle_types
    # where the boxes are our choice of determinant/permanent for each particle type, and between the bra and the ket, have to be the same for all
    # particle types other than the one labeled by ptcl_no (the argument, particle_no).
    # So we will iterate over all possible choices of determinants for the other particles

    # Lists of the row and column indices and the values (will all be the same, value) in the Hamiltonian matrix.
    mtx_rows = []
    mtx_cols = []
    mtx_values = []

    # Iterate all over choices of determinant/permanent for the particles coming BEFORE ptcl1_no
    for i in range(bases[particle1_no]):
        # Iterate all over choices of determinant/permanent for the particles coming AFTER ptcl1_no but BEFORE ptcl2_no
        for j in range(int(bases[particle2_no]/bases[particle1_no+1])):
            # Iterate all over choices of determinant/permanent for the particles coming AFTER ptcl2_no
            for k in range(int(total_states/bases[particle2_no+1])):
                # Construct the indices in the Hamiltonian matrix (base convention) for these bra and ket states
                bra_idx = i + (bra1 * bases[particle1_no]) + (j * bases[particle1_no+1]) + (bra2 * bases[particle2_no]) + (k * bases[particle2_no+1])
                ket_idx = i + (ket1 * bases[particle1_no]) + (j * bases[particle1_no+1]) + (ket2 * bases[particle2_no]) + (k * bases[particle2_no+1])

                # Add to the list of elements
                mtx_rows.append(bra_idx)
                mtx_cols.append(ket_idx)
                mtx_values.append(value)

    return coo_matrix((mtx_values, (mtx_rows, mtx_cols)), shape=(total_states, total_states)).tocsr()

# For full FCI, we will construct the Hamiltonian in the subspace of all states with only the correct particle numbers
# A basis for this space is constructed from N-particle determinants/permanents of the one-particle basis states for correct N

# Total number of states in this space
total_states = 1

# Number of permanents/determinants for each particle.
no_states = []

# For the Hamiltonian matrix, we will index the states as follows:
# Let N_i be the number of states for the i-th particle
# The state formed from the c_0 - th particle 1 permanent/determinant, c_1 - th particle 2 perminant/determinant, etc (c_i's zero indexed)
# Will get the index (c_0) + (c_1 * N_0) + (c_2 * N_1 * N_0) + (c_3 * N_2 * N_1 * N_0) + ...

# This works basically like a numeral base system where each digit has a different base (each digit is a particle).
# This array will contain the bases for each particle: [1, N_0, N_1 * N_0, ...]
bases = []

for symb in particles:
    particles[symb]['states'] = []

    # Construct all N-particle states for the correct N, in the forms of arrays of 1s and 0s. They will be indexed in the order we get them from itertools
    # States with the same spatial wave function will be grouped together. For example, for 4 spatial orbitals with A and B spin states, the significance of the bits will be
    # Bit number:       12345678
    # Spin:             ABABABAB
    # Spatial orbital:  11223344
    for indices in itertools.combinations(range(particles[symb]['no_spin_orbitals']), particles[symb]['count']):
        array = [0] * particles[symb]['no_spin_orbitals']
        for index in indices:
            array[index] = 1

        particles[symb]['states'].append(array)

    particles[symb]['no_states'] = len(particles[symb]['states'])
    particles[symb]['base'] = total_states

    no_states.append(particles[symb]['no_states'])
    bases.append(particles[symb]['base'])
    total_states *= particles[symb]['no_states']

bases.append(total_states) # Having this extra element will be helpful in construct_1_particle_interaction



# We will construct the Hamiltonian matrix in this basis, one interaction at a time

t_mtx = csr_matrix((total_states,total_states)) # Create empty sparse matrix of the appropriate size for the KE operator

# One-body "interactions" (kinetic energy)
for particle_no in range(particle_types):
    symb = particle_names[particle_no]
    particle = particles[symb]

    mass = particle['properties']['mass'] # Mass of the particle
    spin = particle['properties']['spin'] # Spin of the particle

    # Get kinetic energy integrals
    ke_int = kinetic_energy_integral(particle['basis'], particle['transform']) / mass
    particle['ke_int'] = ke_int # Store them

    # Get the spin orbital integrals
    ke_int_spin = IntegralSpinWrapper(ke_int, spin)


    for bra_idx in range(particle['no_states']):
        bra = particle['states'][bra_idx]

        # Indices of the occupied & unoccupied one-particle states for this bra
        occupied = [i for i in range(len(bra)) if bra[i] == 1]
        unoccupied = [i for i in range(len(bra)) if bra[i] == 0]

        print(occupied, unoccupied)

        # First, the element <XXXbraYYY|T|XXXbraYYY> (we will consider all choices of states for the other particles, XXX YYY, by calling construct_1_particle_interaction).
        # That is, the bra and the ket are the same state
        # According to Szabo, this is sum_i [i | O_1 | i] for all occupied orbitals i.
        bra_bra_elmt = 0

        for i in occupied: # Iterate through all occupied one-particle states
            bra_bra_elmt += ke_int_spin[i, i] # Add [i | KE | i]

        t_mtx += construct_1_particle_interaction(particle_no, bra_idx, bra_idx, bra_bra_elmt)

        # Now, we consider elements of the form <XXXbraYYY|T|XXXketYYY> for all ket. Ket must differ from bra by at most one one-particle state,
        # So we will iterate over all states that differ by only one one-particle state.
        # Szabo says this matrix element will be [i | KE | j], where i is the one one-particle state that's only in the bra, and j is the state that's only in the ket
        for bra_1p in occupied: # Iterate through all occupied one-particle states. This is the state we will not include in the ket
            for ket_1p in unoccupied: # This is the state we will replace bra_1p with in our ket state
                # Construct ket state
                ket = bra.copy()
                ket[bra_1p] = 0
                ket[ket_1p] = 1

                #print(ket, bra_1p, ket_1p)

                ket_idx = particle['states'].index(ket)

                bra_ket_elmt = ke_int_spin[bra_1p, ket_1p]
                #print(bra_ket_elmt)

                t_mtx += construct_1_particle_interaction(particle_no, bra_idx, ket_idx, bra_ket_elmt)


        #print(bra)
        #print(h_mtx)

        if(bra_idx > 0):
            break

v_mtx = csr_matrix((total_states,total_states))

# Two-body interactions between particles of the SAME type. TODO: implement boson interactions with their different exchange behavior
for particle_no in range(particle_types):
    print("--------------", particle_no, "----------------")
    symb = particle_names[particle_no]
    particle = particles[symb]

    spin = particle['properties']['spin'] # Spin of the particle

    # Get two body integrals (coulomb force)
    cmb_int = electron_repulsion_integral(particle['basis'], particle['transform'], notation='chemist') * (particle['properties']['charge'] ** 2)
    particle['cmb_int'] = cmb_int # Store them

    # Get the spin orbital integrals
    cmb_int_spin = IntegralSpinWrapper(cmb_int, (spin, spin))

    # Iterate over all bra N-particle states for this particle
    for bra_idx in range(particle['no_states']):
        bra = particle['states'][bra_idx]

        # Indices of the occupied & unoccupied one-particle states for this bra
        occupied = [i for i in range(len(bra)) if bra[i] == 1]
        unoccupied = [i for i in range(len(bra)) if bra[i] == 0]

        print("~~~~~~~", occupied)

        # First, the element <XXXbraYYY|V|XXXbraYYY> (we will consider all choices of states for the other particles, XXX YYY, by callling construct_1_particle_interaction.
        # That is, the bra and the kets are the same state
        # According to Szabo, this is sum_i sum_{j<i} [ii|jj] - [ij|ji] (chemists notation)
        bra_bra_elmt = 0

        for i in range(len(occupied)):
            for j in range(i):
                i_state = occupied[i]
                j_state = occupied[j]

                print(i_state, j_state)

                bra_bra_elmt += cmb_int_spin[i,i,j,j] - cmb_int_spin[i,j,j,i]

        v_mtx += construct_1_particle_interaction(particle_no, bra_idx, bra_idx, bra_bra_elmt)

        # Now for states where the bra differs by only one one-particle state:
        for bra_1p in occupied:
            for ket_1p in unoccupied:
                # Construct ket state
                ket = bra.copy()
                ket[bra_1p] = 0
                ket[ket_1p] = 1

                # Get its index
                ket_idx = particle['states'].index(ket)

                # We require the determinants to have the same ordering of states except for the one state that's different. Since this won't be the case, we need to include a parity factor
                # Number of permutations required to align the states = number of states in COMMON between the two different states
                perms = sum(min(bra_1p, ket_1p) < o < max(bra_1p, ket_1p) for o in occupied)

                # Parity obtained after aligning the states up
                parity = 1 if perms % 2 == 0 else -1

                bra_ket_elmt = 0

                for o in occupied:
                    # Only iterate over states that are in common between the two
                    if o != bra_1p:
                        bra_ket_elmt += cmb_int_spin[bra_1p, ket_1p, o, o] - cmb_int_spin[bra_1p, o, o, ket_1p]

                bra_ket_elmt *= parity

                v_mtx += construct_1_particle_interaction(particle_no, bra_idx, ket_idx, bra_ket_elmt)

        # Now for states where the bra differs from the ket by two one-particle states:
        for bra1_1p, bra2_1p in itertools.combinations(occupied, 2):
            for ket1_1p, ket2_1p in itertools.combinations(unoccupied, 2):
                print(bra1_1p, bra2_1p, '----', ket1_1p, ket2_1p)

                perms = sum(min(bra1_1p, ket1_1p) < o < max(bra1_1p, ket1_1p) for o in occupied) + sum(min(bra2_1p, ket2_1p) < o < max(bra2_1p, ket2_1p) for o in occupied)

                parity = 1 if perms % 2 == 0 else -1

                # Construct ket
                ket = bra.copy()
                ket[bra1_1p] = 0
                ket[bra2_1p] = 0
                ket[ket1_1p] = 1
                ket[ket2_1p] = 1

                ket_idx = particle['states'].index(ket)

                bra_ket_elmt = parity * (cmb_int_spin[bra1_1p, ket1_1p, bra2_1p, ket2_1p] - cmb_int_spin[bra1_1p, ket2_1p, bra2_1p, ket1_1p])

                v_mtx += construct_1_particle_interaction(particle_no, bra_idx, ket_idx, bra_ket_elmt)


        if(bra_idx > 0):
            break

# TODO: I can definitely put some of this stuff in neater functions. Like a function that takes a bra index and spits out all bra indices differing by 1 or 2 states, along with which states were changed and what the parity is. Grouped together in tuples!

# Two-body interactions between particles of DIFFERENT type.
for particle2_no in range(particle_types):
    for particle1_no in range(particle2_no): # so that particle1_no < particle2_no
        symb1 = particle_names[particle1_no]
        symb2 = particle_names[particle2_no]
        print("--------------", particle1_no, symb1, ',', particle2_no, symb2, "----------------")
        particle1 = particles[symb1]
        particle2 = particles[symb2]

        spin1 = particle1['properties']['spin'] # Spin of the particle
        spin2 = particle2['properties']['spin'] #

        b1 = particle1['no_spatial_orbitals'] # number of spatial basis functions for each particle
        b2 = particle2['no_spatial_orbitals']

        # Get two body integrals (coulomb force). Have to combine the bases and then select only the integrals between the two particles
        cmb_int = electron_repulsion_integral(particle1['basis'] + particle2['basis'], transform=block_diag(particle1['transform'], particle2['transform']), notation='chemist')[0:b1, 0:b1, b1:b1+b2, b1:b1+b2]
        cmb_int *= particle1['properties']['charge'] * particle2['properties']['charge']
        #particle['cmb_int'] = cmb_int # Store them

        # Get the spin orbital integrals
        cmb_int_spin = IntegralSpinWrapper(cmb_int, (spin1, spin2))

        # Iterate over all bra N-particle states for these 2 particles
        for bra1_idx in range(particle1['no_states']):
            bra1 = particle1['states'][bra1_idx]

            # Indices of the occupied & unoccupied one-particle states for this bra
            occupied1 = [i for i in range(len(bra1)) if bra1[i] == 1]
            unoccupied1 = [i for i in range(len(bra1)) if bra1[i] == 0]

            for bra2_idx in range(particle2['no_states']):
                bra2 = particle2['states'][bra2_idx]

                # Indices of the occupied & unoccupied one-particle states for this bra
                occupied2 = [i for i in range(len(bra2)) if bra2[i] == 1]
                unoccupied2 = [i for i in range(len(bra2)) if bra2[i] == 0]

            bra_bra_elmt = 0

            # Rule if all states are the same between bra and ket: sum_i,j [ii|jj] for all occupied states i, j
            for o1 in occupied1:
                for o2 in occupied2:
                    bra_bra_elmt += cmb_int_spin[o1, o1, o2, o2]

            v_mtx += construct_2_particle_interaction(particle1_no, particle2_no, o1, o1, o2, o2, bra_bra_elmt)



[0] [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
[1] [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
[0] [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]
[1] [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]
[0, 1, 2, 3] [4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
[0, 1, 2, 4] [3, 5, 6, 7, 8, 9, 10, 11, 12, 13]
-------------- 0 ----------------
~~~~~~~ [0]
~~~~~~~ [1]
-------------- 1 ----------------
~~~~~~~ [0]
~~~~~~~ [1]
-------------- 2 ----------------
~~~~~~~ [0, 1, 2, 3]
1 0
2 0
2 1
3 0
3 1
3 2
0 1 ---- 4 5
0 1 ---- 4 6
0 1 ---- 4 7
0 1 ---- 4 8
0 1 ---- 4 9
0 1 ---- 4 10
0 1 ---- 4 11
0 1 ---- 4 12
0 1 ---- 4 13
0 1 ---- 5 6
0 1 ---- 5 7
0 1 ---- 5 8
0 1 ---- 5 9
0 1 ---- 5 10
0 1 ---- 5 11
0 1 ---- 5 12
0 1 ---- 5 13
0 1 ---- 6 7
0 1 ---- 6 8
0 1 ---- 6 9
0 1 ---- 6 10
0 1 ---- 6 11
0 1 ---- 6 12
0 1 ---- 6 13
0 1 ---- 7 8
0 1 ---- 7 9
0 1 ---

In [103]:
zuprint(v_mtx)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 512 stored elements and shape (512512, 512512)>
  Coords	Values
  (0, 0)	3.4914122245770276
  (1, 1)	3.4914122245770276
  (2, 2)	3.4914122245770276
  (3, 3)	3.4914122245770276
  (4, 4)	3.4914122245770276
  (5, 5)	3.4914122245770276
  (6, 6)	3.4914122245770276
  (7, 7)	3.4914122245770276
  (8, 8)	3.4914122245770276
  (9, 9)	3.4914122245770276
  (10, 10)	3.4914122245770276
  (11, 11)	3.4914122245770276
  (12, 12)	3.4914122245770276
  (13, 13)	3.4914122245770276
  (14, 14)	3.4914122245770276
  (15, 15)	3.4914122245770276
  (16, 16)	3.4914122245770276
  (17, 17)	3.4914122245770276
  (18, 18)	3.4914122245770276
  (19, 19)	3.4914122245770276
  (20, 20)	3.4914122245770276
  (21, 21)	3.4914122245770276
  (22, 22)	3.4914122245770276
  (23, 23)	3.4914122245770276
  (24, 24)	3.4914122245770276
  :	:
  (487, 487)	3.4914122245770276
  (488, 488)	3.4914122245770276
  (489, 489)	3.4914122245770276
  (490, 490)	3.4914122245770276
  (491, 49

In [98]:
print(v_mtx[0,0])
print(cmb_int[0,0,0,0] + cmb_int[1,1,1,1] + 4*cmb_int[0,0,1,1] - 2*
    cmb_int[0,1,1,0])

3.4914122245770276
3.4914122245770276


In [92]:
8#print(h_mtx[2,2])
H_ke = kinetic_energy_integral(particles['H']['basis'], particles['H']['transform'])
Li_ke = kinetic_energy_integral(particles['Li']['basis'], particles['Li']['transform'])
e_ke = kinetic_energy_integral(particles['e']['basis'], particles['e']['transform'])

H_mass = particles['H']['properties']['mass']
Li_mass = particles['Li']['properties']['mass']
e_mass = particles['e']['properties']['mass']

c = H_ke[0,0] / H_mass + Li_ke[0,0] / Li_mass + 2*(e_ke[0,0] + e_ke[1,1]) / e_mass
d = H_ke[0,1] / H_mass
e = H_ke[1,1] / H_mass + Li_ke[0,0] / Li_mass + 2*(e_ke[0,0] + e_ke[1,1]) / e_mass

f = 4 * (particles['H']['cmb_int'][0,0,0,0] - particles['H']['cmb_int'][0,1,1,0]
print(f)

0.13663128710382771


In [88]:
print()

2.1483403907432796


In [20]:
print()

2


In [27]:
print(particles.keys())

dict_keys(['H', 'Li', 'e'])


In [31]:
print(particles['e']['states'][0])

[1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [13]:
print(len(ke_int.shape))

2


In [150]:
prendus = construct_2_particle_interaction(0,2,3,1,0,1,5).tocoo()
[(base_breakdown(row), base_breakdown(col)) for row, col in list(zip(prendus.row, prendus.col))]

[([3, 0, 1], [0, 0, 1]),
 ([3, 1, 1], [0, 1, 1]),
 ([3, 2, 1], [0, 2, 1]),
 ([3, 3, 1], [0, 3, 1]),
 ([3, 4, 1], [0, 4, 1]),
 ([3, 5, 1], [0, 5, 1]),
 ([3, 6, 1], [0, 6, 1]),
 ([3, 7, 1], [0, 7, 1]),
 ([3, 8, 1], [0, 8, 1]),
 ([3, 9, 1], [0, 9, 1]),
 ([3, 10, 1], [0, 10, 1]),
 ([3, 11, 1], [0, 11, 1]),
 ([3, 12, 1], [0, 12, 1]),
 ([3, 13, 1], [0, 13, 1]),
 ([3, 14, 1], [0, 14, 1]),
 ([3, 15, 1], [0, 15, 1]),
 ([3, 16, 1], [0, 16, 1]),
 ([3, 17, 1], [0, 17, 1]),
 ([3, 18, 1], [0, 18, 1]),
 ([3, 19, 1], [0, 19, 1]),
 ([3, 20, 1], [0, 20, 1]),
 ([3, 21, 1], [0, 21, 1]),
 ([3, 22, 1], [0, 22, 1]),
 ([3, 23, 1], [0, 23, 1]),
 ([3, 24, 1], [0, 24, 1]),
 ([3, 25, 1], [0, 25, 1]),
 ([3, 26, 1], [0, 26, 1]),
 ([3, 27, 1], [0, 27, 1]),
 ([3, 28, 1], [0, 28, 1]),
 ([3, 29, 1], [0, 29, 1]),
 ([3, 30, 1], [0, 30, 1]),
 ([3, 31, 1], [0, 31, 1])]

In [123]:
# base
def base_breakdown(x):
    return [int((x % bases[i+1])/bases[i]) for i in range(particle_types)]

In [165]:
particle1 = particles['H']
particle2 = particles['Li']

b1 = particle1['no_spatial_orbitals'] # number of spatial basis functions for each particle
b2 = particle2['no_spatial_orbitals']

cmb_int = electron_repulsion_integral(particle1['basis'] + particle2['basis'], transform=block_diag(particle1['transform'], particle2['transform']), notation='chemist')[0:b1, 0:b1, b1:b1+b2, b1:b1+b2]
cmb_int *= particle1['properties']['charge'] * particle2['properties']['charge']

print(cmb_int.shape)
print(cmb_int[0,0,0,0])


(8, 8, 8, 8)
0.9953176380940549
-3


In [ ]:
par